In [1]:


import pandas as pd
import numpy as np
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

nltk.download('stopwords')
nltk.download('wordnet')


#Load the dataset
df = pd.read_csv("IMDB Dataset.csv")

#view the dataset
print(df.head())
print(df.shape)
df.info()
print(df['sentiment'].value_counts())

#text processing
stop_words = set(stopwords.words('english'))

# Keep important negation words
stop_words.discard('not')
stop_words.discard('no')
stop_words.discard('nor')

lemmatizer = WordNetLemmatizer()

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenize
    words = text.split()

    # Remove stopwords except not/no/nor
    words = [word for word in words if word not in stop_words]

    # Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

df['clean_review'] = df['review'].apply(clean_text)

df['sentiment'] = df['sentiment'].map({
    'positive':1,
    'negative':0
})

print(df.head())


#TF-IDF Vectorization

tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1,2)
)
X = tfidf.fit_transform(df['clean_review'])
y = df['sentiment']


#split dataset

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

#train the model

model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)

#prediction

y_pred = model.predict(X_test)

#accuracy

accuracy = accuracy_score(y_test,y_pred)
print("Accuracy :", accuracy)

#precision

precision = precision_score(y_test, y_pred)
print("Precision :", precision)

#recall
recall = recall_score(y_test,y_pred)
print("Recall :", recall)

#F1 score
f1 = f1_score(y_test,y_pred)
print("F1 score :", f1)

#confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

#classification report
print(classification_report(y_test,y_pred))

print("Model training completed successfully!")

#Predict new reviews

review = input("Enter a movie review: ")

clean_review = clean_text(review)

print("Clean Review :", clean_review)

vector = tfidf.transform([clean_review])

prediction = model.predict(vector)

if prediction[0] == 1:
    print("Predicted Sentiment : Positive")
else:
    print("Predicted Sentiment : Negative")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Divya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Divya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
                                              review  sentiment  \
0  One of the other reviewers has mentioned that ...          1   
1  A wonderful little production. <br /><br />The...          1   
2  I thought this was a wonderfu

Enter a movie review:  the movie is not interesting


Clean Review : movie not interesting
Predicted Sentiment : Negative
